In [0]:
# =============================================================================
# Silver · dim_combustivel
# Fonte: mapeamento manual (seed) baseado nos 29 tipos distintos do frota_raw
# =============================================================================
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%sql
USE CATALOG brazil_car_fleet;
USE SCHEMA silver;

In [0]:
# -----------------------------------------------------------------------------
# 1. Seed table: mapeamento completo nm_combustivel → nm_grupo + fl_identificado
#    Inclui todos os 29 tipos distintos encontrados na bronze.brazi_car_fleet
# -----------------------------------------------------------------------------
 
combustivel_data = [
    # (nm_combustivel,                    nm_grupo,               fl_identificado)
 
    # --- Elétrico Puro ---
    ("ELETRICO",                          "Elétrico Puro",        True),
    ("ELETRICO/FONTE EXTERNA",            "Elétrico Puro",        True),
    ("ELETRICO/FONTE INTERNA",            "Elétrico Puro",        True),
    ("CELULA COMBUSTIVEL",                "Elétrico Puro",        True),
 
    # --- Híbrido ---
    ("HIBRIDO",                           "Híbrido",              True),
    ("HIBRIDO PLUG-IN",                   "Híbrido",              True),
    ("GASOLINA/ELETRICO",                 "Híbrido",              True),
    ("DIESEL/ELETRICO",                   "Híbrido",              True),
    ("ETANOL/ELETRICO",                   "Híbrido",              True),
    ("GASOLINA/ALCOOL/ELETRICO",          "Híbrido",              True),
 
    # --- Flex ---
    ("ALCOOL/GASOLINA",                   "Flex",                 True),
    ("GASOLINA/ALCOOL/GAS NATURAL",       "Flex",                 True),
    ("GASOL/GAS NATURAL COMBUSTIVEL",     "Flex",                 True),
    ("GASOLINA/GAS NATURAL VEICULAR",     "Flex",                 True),
    ("ALCOOL/GAS NATURAL COMBUSTIVEL",    "Flex",                 True),
    ("ALCOOL/GAS NATURAL VEICULAR",       "Flex",                 True),
    ("DIESEL/GAS NATURAL VEICULAR",       "Flex",                 True),
    ("DIESEL/GAS NATURAL COMBUSTIVEL",    "Flex",                 True),
 
    # --- Combustão Fóssil ---
    ("GASOLINA",                          "Combustão Fóssil",     True),
    ("DIESEL",                            "Combustão Fóssil",     True),
    ("GAS NATURAL VEICULAR",              "Combustão Fóssil",     True),
    ("GAS METANO",                        "Combustão Fóssil",     True),
    ("GAS/NATURAL/LIQUEFEITO",            "Combustão Fóssil",     True),
    ("GASOGENIO",                         "Combustão Fóssil",     True),
 
    # --- Combustão Renovável ---
    ("ALCOOL",                            "Combustão Renovável",  True),
 
    # --- Não Identificado ---
    ("Sem Informação",                    "Não Identificado",     False),
    ("VIDE/CAMPO/OBSERVACAO",             "Não Identificado",     False),
    ("Não Identificado",                  "Não Identificado",     False),
    ("Não se Aplica",                     "Não Identificado",     False),
]
 
df_seed = spark.createDataFrame(
    combustivel_data,
    schema=["nm_combustivel", "nm_grupo", "fl_identificado"]
)

In [0]:
# -----------------------------------------------------------------------------
# 2. Auditoria: verificar se algum tipo do brazil_car_fleet não está mapeado no seed
#    Se aparecer algo aqui, adicione manualmente ao seed acima antes de salvar
# -----------------------------------------------------------------------------
 
df_raw_tipos = (
    spark.table("bronze.brazil_car_fleet")
    .select("nm_combustivel")
    .distinct()
)
 
df_sem_mapeamento = df_raw_tipos.join(
    df_seed.select("nm_combustivel"),
    on="nm_combustivel",
    how="left_anti"  # retorna apenas os que NÃO estão no seed
)
 
qtd_sem_mapeamento = df_sem_mapeamento.count()
 
if qtd_sem_mapeamento > 0:
    print(f"⚠️  {qtd_sem_mapeamento} tipo(s) sem mapeamento — adicione ao seed antes de continuar:")
    df_sem_mapeamento.show(truncate=False)
else:
    print("✅ Todos os tipos de combustível estão mapeados.")

In [0]:
# -----------------------------------------------------------------------------
# 3. Gerar surrogate key e montar dim_combustivel final
# -----------------------------------------------------------------------------
 
df_dim_combustivel = (
    df_seed
    .withColumn(
        "id_combustivel",
        F.row_number().over(Window.orderBy("nm_grupo", "nm_combustivel"))
    )
    .select(
        "id_combustivel",
        "nm_combustivel",
        "nm_grupo",
        "fl_identificado",
    )
)

In [0]:
# -----------------------------------------------------------------------------
# 4. Salvar na Silver como Delta
# -----------------------------------------------------------------------------
 
(
    df_dim_combustivel
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_combustivel")
)
 
print("\nsilver.dim_combustivel salva com sucesso.")
df_dim_combustivel.orderBy("nm_grupo", "nm_combustivel").show(40, truncate=False)